In [41]:
import os
from dotenv import load_dotenv
from bardapi import Bard
import anthropic
from IPython.display import Markdown, display, update_display

import time
import google.generativeai as genai

In [51]:
# Load biến môi trường từ file .env
load_dotenv(override=True)

# Lấy API key
bard_api_key = os.getenv('BARD_API_KEY')

# Gán key và khởi tạo Bard nếu có
if bard_api_key:
    os.environ['_BARD_API_KEY'] = bard_api_key
    bard = Bard()
    print(f"Bard API Key loaded and begins with: {bard_api_key[:8]}")
else:
    print("Bard API Key not set")


genai.configure(api_key=bard_api_key)

Bard API Key loaded and begins with: AIzaSyB6


In [57]:
def message_gpt(text):
    try:
        # Chọn mô hình phù hợp từ danh sách
        model = genai.GenerativeModel("gemini-1.5-pro")  # Ví dụ sử dụng gemini-1.5-pro
        response = model.generate_content(text)
        return response.text 
    except Exception as e:
        return f"Error: {str(e)}"


In [15]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

shout("hello")

Shout has been called with input hello


'HELLO'

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox").launch()
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

In [27]:
view1 = gr.Interface(
    fn=shout,
    inputs=[gr.Textbox(label="Your message:", lines=6)],
    outputs=[gr.Textbox(label="Response:", lines=8)],
    flagging_mode="never"
)
view1.launch()

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [60]:
view2 = gr.Interface(
    fn=message_gpt,
    inputs=[gr.Textbox(label="Your message:", lines=6)],
    outputs=[gr.Textbox(label="Response:", lines=8)],
    flagging_mode="never"
)
view2.launch()

* Running on local URL:  http://127.0.0.1:7866

To create a public link, set `share=True` in `launch()`.


In [64]:
system_message = "You are a helpful assistant that responds in markdown"

view = gr.Interface(
    fn=message_gpt,
    inputs=[gr.Textbox(label="Your message:", lines=6)],
    outputs=[gr.Textbox(label="Response:", lines=8)],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7868

To create a public link, set `share=True` in `launch()`.


In [90]:
def stream_gpt(prompt):
    try:
        model = genai.GenerativeModel("gemini-1.5-pro")
        response = model.generate_content(prompt, stream=True)
        full_response = ""
        for chunk in response:
            full_response += chunk.text
            yield full_response
    except Exception as e:
        yield f"Error: {str(e)}"


In [92]:
def stream_claude(prompt):
    try:
        model = genai.GenerativeModel("gemini-1.5-pro-001")
        response = model.generate_content(prompt, stream=True)
        full_response = ""
        for chunk in response:
            full_response += chunk.text
            yield full_response
    except Exception as e:
        yield f"Error: {str(e)}"


In [94]:
def stream_model(prompt, model):
    if model == "Gemini-1.5-Pro":
        result = stream_gpt(prompt)  # Sử dụng stream_gpt cho Gemini-1.5-Pro
    elif model == "Gemini-1.5-Pro-001":
        result = stream_claude(prompt)  # Sử dụng stream_claude cho Gemini-1.5-Pro-001
    else:
        raise ValueError("Unknown model")
    yield from result

# Giao diện Gradio
view = gr.Interface(
    fn=stream_model,
    inputs=[
        gr.Textbox(label="Your message:"), 
        gr.Dropdown(["Gemini-1.5-Pro", "Gemini-1.5-Pro-001"], label="Select model", value="Gemini-1.5-Pro")
    ],
    outputs=[gr.Textbox(label="Response:", lines=8)],
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7875

To create a public link, set `share=True` in `launch()`.


In [72]:
class Website:
    url: str
    title: str
    text: str

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [76]:
system_message = "You are an assistant that analyzes the contents of a company website landing page \
and creates a short brochure about the company for prospective customers, investors, and recruits. Respond in markdown."

def stream_brochure(company_name, url, model):
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += Website(url).get_contents()
    
    # Chọn mô hình Gemini thay vì GPT hoặc Claude
    if model == "Gemini-1.5-Pro":
        result = stream_gpt(prompt)  # Sử dụng stream_gpt cho Gemini-1.5-Pro
    elif model == "Gemini-1.5-Pro-001":
        result = stream_claude(prompt)  # Sử dụng stream_claude cho Gemini-1.5-Pro-001
    else:
        raise ValueError("Unknown model")


In [78]:
view = gr.Interface(
    fn=stream_brochure,
    inputs=[
        gr.Textbox(label="Company name:"),
        gr.Textbox(label="Landing page URL including http:// or https://"),
        gr.Dropdown(["Gemini-1.5-Pro", "Gemini-1.5-Pro-001"], label="Select model")  # Chọn mô hình Gemini
    ],
    outputs=[gr.Markdown(label="Brochure:")],
    flagging_mode="never"
)
view.launch()


* Running on local URL:  http://127.0.0.1:7872

To create a public link, set `share=True` in `launch()`.
